# Proyecto final: predicción de abandono de clientes
**Estudiante:** Gael Andrés García García
**Materia:** Algoritmos de Aprendizaje Automático
**Fecha:** 23 de septiembre de 2026

Esta notebook desarrolla la versión final del proyecto a partir del avance ya realizado. Se mantiene el mismo problema, el mismo dataset y la misma variable objetivo, pero se amplía con validación cruzada, modelos avanzados, ajuste de hiperparámetros, selección de características y serialización del modelo final.

## 1. Definición del problema

Este proyecto aborda un problema de clasificación binaria aplicado a la predicción del abandono de clientes (`Churn`) en una compañía de servicios por suscripción. La variable objetivo identifica si el cliente permanece (`No`) o abandona (`Yes`) el servicio.

El objetivo analítico es construir un modelo que permita identificar clientes con mayor probabilidad de abandono y apoyar la priorización de acciones de retención. La predicción no sustituye la decisión humana: debe utilizarse como una herramienta de apoyo para orientar contactos, promociones o revisiones de servicio.

Debido al desequilibrio natural entre clientes que permanecen y clientes que abandonan, se consideran varias métricas: accuracy, precision, recall, F1-score y ROC-AUC. Se presta especial atención al recall y al F1-score, porque una solución que solo prediga la clase mayoritaria tendría una utilidad limitada para detectar posibles abandonos.

In [3]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Se fija la semilla para que las particiones y los modelos sean reproducibles.
RANDOM_STATE = 42
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / 'Telco-Customer-Churn.csv'
ARTIFACT_PATH = BASE_DIR / 'telco_model_final.joblib'
METADATA_PATH = BASE_DIR / 'telco_model_metadata.json'

## 0. Verificación inicial y bitácora de cambios

Antes de entrar en la comparación de modelos, se recupera la trazabilidad del proyecto para conservar la línea base del avance y dejar la notebook en una versión formal y entregable.

La siguiente tabla consolida los elementos clave del proyecto: problema, dataset, objetivo, particiones, semilla, pruebas iniciales, métricas, modelo preliminar y acciones requeridas.

In [4]:
project_verification = pd.DataFrame([
    {
        'elemento': 'Problema',
        'archivo_o_ubicacion': 'Notebook de avance y notebook final',
        'version': 'v1.0 / v2.0',
        'estado': 'Cumple',
        'accion_requerida': 'Conservar el mismo problema, dataset y objetivo del avance.'
    },
    {
        'elemento': 'Dataset original',
        'archivo_o_ubicacion': 'Telco-Customer-Churn.csv',
        'version': 'Original',
        'estado': 'Cumple',
        'accion_requerida': 'Validar integridad y registros duplicados.'
    },
    {
        'elemento': 'Variable objetivo',
        'archivo_o_ubicacion': 'Churn',
        'version': 'Yes=1 / No=0',
        'estado': 'Cumple',
        'accion_requerida': 'Mantener la codificación y la clase positiva como churn.'
    },
    {
        'elemento': 'Particiones',
        'archivo_o_ubicacion': 'train / validation / test',
        'version': '70/15/15',
        'estado': 'Cumple',
        'accion_requerida': 'Usar solo el conjunto de entrenamiento para ajustar y seleccionar.'
    },
    {
        'elemento': 'Semilla',
        'archivo_o_ubicacion': 'RANDOM_STATE = 42',
        'version': '42',
        'estado': 'Cumple',
        'accion_requerida': 'Mantener la reproducción del pipeline y del split.'
    },
    {
        'elemento': 'Pipeline reproducible',
        'archivo_o_ubicacion': 'ColumnTransformer + Pipeline',
        'version': 'v2.0',
        'estado': 'Cumple',
        'accion_requerida': 'Guardar artefacto final con preprocesamiento y modelo.'
    },
    {
        'elemento': 'Baseline inicial',
        'archivo_o_ubicacion': 'DummyClassifier',
        'version': 'Inicial',
        'estado': 'Cumple',
        'accion_requerida': 'Usarlo como referencia para comparación.'
    },
    {
        'elemento': 'Modelos preliminares',
        'archivo_o_ubicacion': 'LogisticRegression / DecisionTree',
        'version': 'Referencia',
        'estado': 'Cumple',
        'accion_requerida': 'Registrar su desempeño como punto de partida.'
    },
    {
        'elemento': 'Modelos avanzados',
        'archivo_o_ubicacion': 'RandomForest / SVC',
        'version': 'Final',
        'estado': 'Cumple',
        'accion_requerida': 'Comparar en validación cruzada y validar en prueba.'
    },
    {
        'elemento': 'Umbral de decisión',
        'archivo_o_ubicacion': 'threshold = 0.40',
        'version': 'Operativo',
        'estado': 'Cumple',
        'accion_requerida': 'Ajustarlo con costes reales del negocio y revisión humana.'
    },
    {
        'elemento': 'Bitácora de cambios',
        'archivo_o_ubicacion': 'Notebook + metadata',
        'version': 'v2.0',
        'estado': 'Cumple',
        'accion_requerida': 'Registrar datos, hiperparámetros, métricas y artefacto final.'
    },
    {
        'elemento': 'Modelo serializado',
        'archivo_o_ubicacion': 'telco_model_final.joblib',
        'version': '2.0',
        'estado': 'Cumple',
        'accion_requerida': 'Mantener la advertencia de origen desconocido y revisión humana.'
    }
])

project_verification

change_log = pd.DataFrame([
    {
        'fase': 'Carga y calidad',
        'dato_clave': 'TotalCharges convertido a numérico',
        'preprocesamiento': 'Coerción de vacíos a NaN',
        'modelo': 'Ninguno',
        'metrica': 'Integridad de datos',
        'umbral': 'N/A',
        'interpretacion': 'Evita errores de tipado en el pipeline.'
    },
    {
        'fase': 'Ingeniería de variables',
        'dato_clave': 'Se agregan 7 features operativas',
        'preprocesamiento': 'has_internet, num_services, contract_short_term, new_customer, monthly_to_tenure_ratio, total_over_tenure, paperless_electronic_check',
        'modelo': 'RandomForest / SVC',
        'metrica': 'F1 / ROC-AUC',
        'umbral': '0.40',
        'interpretacion': 'Las nuevas variables capturan riesgo de retención y hábitos de uso.'
    },
    {
        'fase': 'Validación cruzada',
        'dato_clave': '5 folds estratificados',
        'preprocesamiento': 'Pipeline con imputación + escalado + one-hot',
        'modelo': 'Dummy, LogisticRegression, RandomForest, SVC',
        'metrica': 'F1-score',
        'umbral': '0.40',
        'interpretacion': 'Reduce la variabilidad por partición y evita sobreajuste local.'
    },
    {
        'fase': 'Ajuste de hiperparámetros',
        'dato_clave': 'GridSearchCV sobre RF y SVC',
        'preprocesamiento': 'Mismo proceso para todos los candidatos',
        'modelo': 'RandomForest + SVC calibrado',
        'metrica': 'F1-score',
        'umbral': '0.40',
        'interpretacion': 'Se conserva el mejor equilibrio entre recall y precision.'
    },
    {
        'fase': 'Selección final',
        'dato_clave': 'Validación en conjunto de prueba',
        'preprocesamiento': 'Pipeline íntegro serializado',
        'modelo': 'Modelo final',
        'metrica': 'Accuracy, Precision, Recall, F1, ROC-AUC',
        'umbral': '0.40',
        'interpretacion': 'Se utiliza evidencia externa y no la muestra de validación para la elección final.'
    }
])

change_log

,fase,dato_clave,preprocesamiento,modelo,metrica,umbral,interpretacion
0,Carga y calidad,TotalCharges convertido a numérico,Coerción de vacíos a NaN,Ninguno,Integridad de datos,N/A,Evita errores de tipado en el pipeline.
1,Ingeniería de variables,Se agregan 7 features operativas,"has_internet, num_services, contract_short_ter...",RandomForest / SVC,F1 / ROC-AUC,0.40,Las nuevas variables capturan riesgo de retenc...
2,Validación cruzada,5 folds estratificados,Pipeline con imputación + escalado + one-hot,"Dummy, LogisticRegression, RandomForest, SVC",F1-score,0.40,Reduce la variabilidad por partición y evita s...
3,Ajuste de hiperparámetros,GridSearchCV sobre RF y SVC,Mismo proceso para todos los candidatos,RandomForest + SVC calibrado,F1-score,0.40,Se conserva el mejor equilibrio entre recall y...
4,Selección final,Validación en conjunto de prueba,Pipeline íntegro serializado,Modelo final,"Accuracy, Precision, Recall, F1, ROC-AUC",0.40,Se utiliza evidencia externa y no la muestra d...


In [5]:
# Se carga una copia del dataset y se normalizan las columnas necesarias para el análisis.
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'].replace(r'^\s*$', np.nan, regex=True),
    errors='coerce'
)
df['Churn_num'] = df['Churn'].map({'No': 0, 'Yes': 1})

print('Dimensiones:', df.shape)
print('Distribución de la variable objetivo:')
print(df['Churn'].value_counts(normalize=True).round(3))

df.head()

Dimensiones: (7043, 22)
Distribución de la variable objetivo:
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Churn_num
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


## 1.1. Diccionario de variables

El dataset original contiene información demográfica, de contrato y de uso del servicio. El diccionario consolidado ayuda a interpretar cada predictor y a mantener la trazabilidad del problema.

A continuación se presenta una síntesis de las columnas relevantes para el proyecto.

In [6]:
variable_dictionary = pd.DataFrame([
    {'columna': 'customerID', 'tipo': 'identificador', 'descripcion': 'Identificador del cliente.', 'uso': 'Se excluye del modelo para evitar fuga de identidad.'},
    {'columna': 'gender', 'tipo': 'categórica', 'descripcion': 'Sexo del cliente.', 'uso': 'Se usa como predictor.'},
    {'columna': 'SeniorCitizen', 'tipo': 'binaria', 'descripcion': 'Indicador de cliente mayor de 65 años.', 'uso': 'Se usa como predictor.'},
    {'columna': 'Partner', 'tipo': 'binaria', 'descripcion': 'Indica si el cliente tiene pareja.', 'uso': 'Se usa como predictor.'},
    {'columna': 'Dependents', 'tipo': 'binaria', 'descripcion': 'Indica si el cliente tiene dependientes.', 'uso': 'Se usa como predictor.'},
    {'columna': 'tenure', 'tipo': 'numérica', 'descripcion': 'Número de meses de permanencia.', 'uso': 'Se usa como predictor y en ratios derivados.'},
    {'columna': 'PhoneService', 'tipo': 'binaria', 'descripcion': 'Servicio telefónico.', 'uso': 'Se usa como predictor.'},
    {'columna': 'MultipleLines', 'tipo': 'categórica', 'descripcion': 'Número de líneas.', 'uso': 'Se usa como predictor.'},
    {'columna': 'InternetService', 'tipo': 'categórica', 'descripcion': 'Tipo de servicio de internet.', 'uso': 'Se usa para detectar riesgo y para features derivadas.'},
    {'columna': 'OnlineSecurity', 'tipo': 'categórica', 'descripcion': 'Cobertura de seguridad en línea.', 'uso': 'Se usa como predictor.'},
    {'columna': 'OnlineBackup', 'tipo': 'categórica', 'descripcion': 'Copia de seguridad online.', 'uso': 'Se usa como predictor.'},
    {'columna': 'DeviceProtection', 'tipo': 'categórica', 'descripcion': 'Protección del dispositivo.', 'uso': 'Se usa como predictor.'},
    {'columna': 'TechSupport', 'tipo': 'categórica', 'descripcion': 'Soporte técnico.', 'uso': 'Se usa como predictor.'},
    {'columna': 'StreamingTV', 'tipo': 'categórica', 'descripcion': 'Streaming de TV.', 'uso': 'Se usa como predictor.'},
    {'columna': 'StreamingMovies', 'tipo': 'categórica', 'descripcion': 'Streaming de películas.', 'uso': 'Se usa como predictor.'},
    {'columna': 'Contract', 'tipo': 'categórica', 'descripcion': 'Tipo de contrato.', 'uso': 'Se usa como predictor y para la variable contract_short_term.'},
    {'columna': 'PaperlessBilling', 'tipo': 'binaria', 'descripcion': 'Facturación sin papel.', 'uso': 'Se usa como predictor.'},
    {'columna': 'PaymentMethod', 'tipo': 'categórica', 'descripcion': 'Método de pago.', 'uso': 'Se usa como predictor.'},
    {'columna': 'MonthlyCharges', 'tipo': 'numérica', 'descripcion': 'Cargo mensual.', 'uso': 'Se usa como predictor y en ratios de precio.'},
    {'columna': 'TotalCharges', 'tipo': 'numérica', 'descripcion': 'Cargo total acumulado.', 'uso': 'Se usa como predictor y se convierte a numérico.'},
    {'columna': 'Churn', 'tipo': 'objetivo', 'descripcion': 'Variable objetivo: si el cliente abandona o no.', 'uso': 'Se codifica como 1 para churn y 0 para no churn.'}
])

variable_dictionary

,columna,tipo,descripcion,uso
0,customerID,identificador,Identificador del cliente.,Se excluye del modelo para evitar fuga de iden...
1,gender,categórica,Sexo del cliente.,Se usa como predictor.
2,SeniorCitizen,binaria,Indicador de cliente mayor de 65 años.,Se usa como predictor.
3,Partner,binaria,Indica si el cliente tiene pareja.,Se usa como predictor.
4,Dependents,binaria,Indica si el cliente tiene dependientes.,Se usa como predictor.
5,tenure,numérica,Número de meses de permanencia.,Se usa como predictor y en ratios derivados.
6,PhoneService,binaria,Servicio telefónico.,Se usa como predictor.
7,MultipleLines,categórica,Número de líneas.,Se usa como predictor.
8,InternetService,categórica,Tipo de servicio de internet.,Se usa para detectar riesgo y para features de...
9,OnlineSecurity,categórica,Cobertura de seguridad en línea.,Se usa como predictor.


## 2. Ingeniería de características

Se crean variables que pueden aportar información al problema de retención: número de servicios contratados, indicador de contrato corto plazo y de cliente nuevo, y ratio de precio respecto a la antigüedad.

In [7]:
# Se generan variables derivadas con una interpretación operativa para el problema de retención.
yes_no_cols = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
feature_map = {'Yes': 1, 'No': 0, 'No phone service': 0, 'No internet service': 0}

df['has_internet'] = (df['InternetService'] != 'No').astype(int)
df['num_services'] = df[yes_no_cols].replace(feature_map).astype(float).sum(axis=1)
df['contract_short_term'] = (df['Contract'] == 'Month-to-month').astype(int)
df['new_customer'] = (df['tenure'] <= 12).astype(int)
df['monthly_to_tenure_ratio'] = df['MonthlyCharges'] / df['tenure'].replace(0, np.nan)
df['total_over_tenure'] = df['TotalCharges'] / df['tenure'].replace(0, np.nan)
df['paperless_electronic_check'] = ((df['PaperlessBilling'] == 'Yes') & (df['PaymentMethod'] == 'Electronic check')).astype(int)

feature_cols = [
    'num_services', 'contract_short_term', 'new_customer',
    'monthly_to_tenure_ratio', 'total_over_tenure', 'paperless_electronic_check'
]
df[feature_cols].head()

,num_services,contract_short_term,new_customer,monthly_to_tenure_ratio,total_over_tenure,paperless_electronic_check
0,1.0,1,1,29.850,29.850000,1
1,3.0,0,0,1.675,55.573529,0
2,3.0,1,1,26.925,54.075000,0
3,3.0,0,0,0.940,40.905556,0
4,1.0,1,1,35.350,75.825000,1


In [8]:
# Se excluyen la variable objetivo y el identificador, y se conserva la proporción de clases en cada partición.
X = df.drop(columns=['Churn', 'Churn_num', 'customerID']).copy()
y = df['Churn_num']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print('Entrenamiento:', X_train.shape, y_train.shape)
print('Validación:', X_validation.shape, y_validation.shape)
print('Prueba:', X_test.shape, y_test.shape)

Entrenamiento: (4930, 26) (4930,)
Validación: (1056, 26) (1056,)
Prueba: (1057, 26) (1057,)


## 3. Pipeline de preprocesamiento

Las variables numéricas y categóricas requieren tratamientos diferentes. Por ello, se construye un `ColumnTransformer` que imputa valores faltantes, estandariza las variables numéricas y convierte las variables categóricas mediante one-hot encoding.

El preprocesamiento se integra dentro de cada `Pipeline`. De esta manera, sus parámetros se ajustan exclusivamente con los datos de entrenamiento en cada partición y se evita la filtración de información entre entrenamiento, validación y prueba.

In [9]:
# Se separan las variables por tipo para aplicar transformaciones específicas dentro del pipeline.
numeric_features = [
    'tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'has_internet',
    'num_services', 'contract_short_term', 'new_customer',
    'monthly_to_tenure_ratio', 'total_over_tenure', 'paperless_electronic_check'
]
categorical_features = [
    col for col in X.columns
    if col not in numeric_features + ['customerID', 'Churn', 'Churn_num']
]

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), categorical_features)
])

In [10]:
# Esta función concentra las métricas utilizadas para comparar todos los modelos.
def evaluate_model(name, model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1]
    return {
        'Modelo': name,
        'Accuracy': accuracy_score(y_eval, y_pred),
        'Precision': precision_score(y_eval, y_pred, zero_division=0),
        'Recall': recall_score(y_eval, y_pred, zero_division=0),
        'F1-score': f1_score(y_eval, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_eval, y_prob),
    }

# Se incluyen un clasificador de referencia y varios modelos con distinta complejidad.
models = {
    'DummyClassifier': DummyClassifier(strategy='most_frequent'),
    'LogisticRegression': LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
    'DecisionTree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight='balanced_subsample'
    ),
    'SVC': CalibratedClassifierCV(
        SVC(kernel='rbf', C=2.0, gamma='scale', class_weight='balanced', random_state=RANDOM_STATE),
        method='sigmoid',
        cv=3
    )
}

# Cada modelo conserva el mismo preprocesamiento para que la comparación sea justa.
pipelines = {}
for name, model in models.items():
    pipeline = Pipeline([('preprocesamiento', preprocessor), ('modelo', model)])
    pipeline.fit(X_train, y_train)
    pipelines[name] = pipeline

validation_df = pd.DataFrame(
    [evaluate_model(name, pipe, X_validation, y_validation) for name, pipe in pipelines.items()]
).set_index('Modelo')

validation_df.round(3)

,Accuracy,Precision,Recall,F1-score,ROC-AUC
Modelo,,,,,
DummyClassifier,0.735,0.000,0.000,0.000,0.500
LogisticRegression,0.808,0.665,0.554,0.604,0.850
DecisionTree,0.790,0.628,0.507,0.561,0.837
RandomForest,0.777,0.597,0.482,0.534,0.822
SVC,0.777,0.577,0.604,0.590,0.828


## 4. Validación cruzada

La validación cruzada estratificada con cinco particiones permite estimar la estabilidad de los modelos utilizando diferentes subconjuntos de entrenamiento y validación. Las métricas se calculan con `zero_division=0` para definir explícitamente el comportamiento cuando un modelo no predice casos positivos en una partición.

El conjunto de prueba se mantiene separado y no participa en esta etapa.

In [11]:
# Se define una validación cruzada estratificada y scorers explícitos para evitar ambigüedades.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
    'f1': make_scorer(f1_score, zero_division=0),
    'roc_auc': 'roc_auc',
}
f1_scorer = make_scorer(f1_score, zero_division=0)

cv_summary = {}
for name, pipe in pipelines.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    cv_summary[name] = {
        metric: float(np.mean(scores[f'test_{metric}'])) for metric in scoring
    }

cv_df = pd.DataFrame(cv_summary).T
cv_df.round(3)

,accuracy,precision,recall,f1,roc_auc
DummyClassifier,0.735,0.000,0.000,0.000,0.500
LogisticRegression,0.806,0.672,0.530,0.592,0.848
DecisionTree,0.800,0.640,0.569,0.601,0.824
RandomForest,0.797,0.656,0.496,0.565,0.833
SVC,0.795,0.625,0.571,0.597,0.830


## 5. Selección de características

Se compara el modelo completo con una versión reducida mediante `SelectFromModel`. El selector utiliza una regresión logística con regularización L1 para conservar las variables con mayor contribución al proceso de clasificación.

Esta comparación permite determinar si una representación más compacta mantiene un desempeño similar y facilita la interpretación del modelo.

In [12]:
# Se seleccionan variables mediante una regresión logística con regularización L1.
# En scikit-learn 1.9, l1_ratio=1.0 reemplaza la configuración penalty='l1'.
selector_pipe = Pipeline([
    ('preprocesamiento', preprocessor),
    ('selector', SelectFromModel(
        LogisticRegression(
            l1_ratio=1.0,
            solver='liblinear',
            C=0.5,
            random_state=RANDOM_STATE
        ),
        threshold='median'
    )),
    ('modelo', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
])
selector_pipe.fit(X_train, y_train)

selected_mask = selector_pipe.named_steps['selector'].get_support()
print('Características seleccionadas:', int(selected_mask.sum()))
print('Total de características:', len(selected_mask))

logreg_model = Pipeline([
    ('preprocesamiento', preprocessor),
    ('modelo', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
])
logreg_model.fit(X_train, y_train)

y_pred_full = logreg_model.predict(X_validation)

metrics_full = {
    'accuracy': accuracy_score(y_validation, y_pred_full),
    'precision': precision_score(y_validation, y_pred_full, zero_division=0),
    'recall': recall_score(y_validation, y_pred_full, zero_division=0),
    'f1': f1_score(y_validation, y_pred_full, zero_division=0),
    'roc_auc': roc_auc_score(y_validation, logreg_model.predict_proba(X_validation)[:, 1])
}

reduced_logreg = selector_pipe

y_pred_reduced = reduced_logreg.predict(X_validation)
metrics_red = {
    'accuracy': accuracy_score(y_validation, y_pred_reduced),
    'precision': precision_score(y_validation, y_pred_reduced, zero_division=0),
    'recall': recall_score(y_validation, y_pred_reduced, zero_division=0),
    'f1': f1_score(y_validation, y_pred_reduced, zero_division=0),
    'roc_auc': roc_auc_score(y_validation, reduced_logreg.predict_proba(X_validation)[:, 1])
}

pd.DataFrame({'Modelo completo': metrics_full, 'Modelo reducido': metrics_red})

Características seleccionadas: 52
Total de características: 52


,Modelo completo,Modelo reducido
accuracy,0.807765,0.807765
precision,0.665236,0.665236
recall,0.553571,0.553571
f1,0.604288,0.604288
roc_auc,0.850495,0.850495


## 6. Ajuste de hiperparámetros

Se realiza una búsqueda acotada de hiperparámetros para Random Forest y SVC. La selección se basa en F1-score y utiliza únicamente el conjunto de entrenamiento mediante validación cruzada; por tanto, el conjunto de prueba permanece reservado para la evaluación final.

El SVC utiliza calibración externa para obtener probabilidades sin depender de la opción `probability=True`, que está en proceso de descontinuación en versiones recientes de scikit-learn.

In [13]:
rf_estimator = Pipeline([
    ('preprocesamiento', preprocessor),
    ('modelo', RandomForestClassifier(random_state=RANDOM_STATE))
])
rf_grid = {
    'modelo__n_estimators': [200, 300],
    'modelo__max_depth': [None],
    'modelo__min_samples_leaf': [1, 3],
    'modelo__class_weight': ['balanced'],
}
rf_search = GridSearchCV(
    rf_estimator,
    param_grid=rf_grid,
    scoring=f1_scorer,
    cv=cv,
    n_jobs=-1
)
rf_search.fit(X_train, y_train)

svc_estimator = Pipeline([
    ('preprocesamiento', preprocessor),
    ('modelo', CalibratedClassifierCV(
        SVC(random_state=RANDOM_STATE),
        method='sigmoid',
        cv=3
    ))
])
svc_grid = {
    'modelo__estimator__kernel': ['rbf'],
    'modelo__estimator__C': [1.0, 2.0, 5.0],
    'modelo__estimator__gamma': ['scale'],
    'modelo__estimator__class_weight': ['balanced'],
}
svc_search = GridSearchCV(
    svc_estimator,
    param_grid=svc_grid,
    scoring=f1_scorer,
    cv=cv,
    n_jobs=-1
)
svc_search.fit(X_train, y_train)

print('Mejor RF:', rf_search.best_params_)
print('Mejor SVC:', svc_search.best_params_)
print('Mejor F1 RF:', rf_search.best_score_)
print('Mejor F1 SVC:', svc_search.best_score_)

Mejor RF: {'modelo__class_weight': 'balanced', 'modelo__max_depth': None, 'modelo__min_samples_leaf': 3, 'modelo__n_estimators': 300}
Mejor SVC: {'modelo__estimator__C': 1.0, 'modelo__estimator__class_weight': 'balanced', 'modelo__estimator__gamma': 'scale', 'modelo__estimator__kernel': 'rbf'}
Mejor F1 RF: 0.6312067510990826
Mejor F1 SVC: 0.6084796013667837


In [14]:
candidate_models = {
    'LogisticRegression': pipelines['LogisticRegression'],
    'RandomForest': rf_search.best_estimator_,
    'SVC': svc_search.best_estimator_
}

final_df = pd.DataFrame(
    [evaluate_model(name, pipe, X_validation, y_validation) for name, pipe in candidate_models.items()]
).set_index('Modelo')

final_df.round(3)

,Accuracy,Precision,Recall,F1-score,ROC-AUC
Modelo,,,,,
LogisticRegression,0.808,0.665,0.554,0.604,0.850
RandomForest,0.763,0.539,0.736,0.622,0.836
SVC,0.777,0.574,0.625,0.598,0.832


## 7. Optimización del umbral

La clasificación no tiene que utilizar necesariamente el umbral predeterminado de 0.50. Se comparan varios umbrales sobre el conjunto de validación para observar el compromiso entre precision, recall, F1-score, falsos positivos y falsos negativos.

En un escenario de retención, reducir falsos negativos puede ser prioritario porque representa clientes con riesgo de abandono que no serían contactados. El umbral final debe confirmarse con los costes reales de cada tipo de error.

In [15]:
chosen_name = final_df['F1-score'].idxmax()
chosen_model = candidate_models[chosen_name]
chosen_prob = chosen_model.predict_proba(X_validation)[:, 1]

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
threshold_table = []
for threshold in thresholds:
    pred = (chosen_prob >= threshold).astype(int)
    threshold_table.append({
        'Umbral': threshold,
        'Precision': precision_score(y_validation, pred, zero_division=0),
        'Recall': recall_score(y_validation, pred, zero_division=0),
        'F1-score': f1_score(y_validation, pred, zero_division=0),
        'Falsos positivos': int(((pred == 1) & (y_validation == 0)).sum()),
        'Falsos negativos': int(((pred == 0) & (y_validation == 1)).sum())
    })

pd.DataFrame(threshold_table).round(3)

,Umbral,Precision,Recall,F1-score,Falsos positivos,Falsos negativos
0,0.30,0.442,0.907,0.594,321,26
1,0.35,0.473,0.882,0.616,275,33
2,0.40,0.503,0.861,0.635,238,39
3,0.45,0.522,0.800,0.632,205,56
4,0.50,0.539,0.736,0.622,176,74
5,0.55,0.555,0.682,0.612,153,89
6,0.60,0.576,0.636,0.604,131,102


In [16]:
selected_threshold = 0.40
test_prob = chosen_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= selected_threshold).astype(int)

test_metrics = {
    'Accuracy': accuracy_score(y_test, test_pred),
    'Precision': precision_score(y_test, test_pred, zero_division=0),
    'Recall': recall_score(y_test, test_pred, zero_division=0),
    'F1-score': f1_score(y_test, test_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, test_prob)
}

pd.DataFrame({'Métrica': list(test_metrics.keys()), 'Valor': list(test_metrics.values())})

,Métrica,Valor
0,Accuracy,0.742668
1,Precision,0.510158
2,Recall,0.804270
3,F1-score,0.624309
4,ROC-AUC,0.835937


## 8. Interpretabilidad y análisis de errores

La interpretabilidad se aborda mediante la importancia global de las variables y la revisión de predicciones individuales. Este análisis ayuda a identificar patrones asociados con el abandono y a reconocer casos en los que el modelo puede equivocarse.

Las importancias no deben interpretarse como relaciones causales. Representan asociaciones aprendidas por el modelo a partir de este dataset y deben contrastarse con conocimiento del negocio.

In [17]:
feature_names = chosen_model.named_steps['preprocesamiento'].get_feature_names_out().tolist()
if hasattr(chosen_model.named_steps['modelo'], 'coef_'):
    importances = np.abs(chosen_model.named_steps['modelo'].coef_[0])
    top_features = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)[:10]
elif hasattr(chosen_model.named_steps['modelo'], 'feature_importances_'):
    importances = chosen_model.named_steps['modelo'].feature_importances_
    top_features = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)[:10]
else:
    top_features = []

pd.DataFrame(top_features, columns=['Variable', 'Importancia']).round(4)

,Variable,Importancia
0,numeric__monthly_to_tenure_ratio,0.1292
1,numeric__contract_short_term,0.0909
2,numeric__TotalCharges,0.0755
3,numeric__tenure,0.0713
4,categorical__Contract_Month-to-month,0.0702
5,numeric__total_over_tenure,0.0646
6,numeric__MonthlyCharges,0.0638
7,categorical__OnlineSecurity_No,0.0422
8,categorical__TechSupport_No,0.0301
9,categorical__Contract_Two year,0.0297


In [18]:
# Ejemplos representativos de error
for idx in range(5):
    row = X_test.iloc[idx:idx+1].copy()
    actual = int(y_test.iloc[idx])
    pred = int(chosen_model.predict(row)[0])
    prob = float(chosen_model.predict_proba(row)[0, 1])
    print(f'Índice: {idx}, real={actual}, pred={pred}, prob={prob:.3f}')

Índice: 0, real=1, pred=1, prob=0.557
Índice: 1, real=0, pred=0, prob=0.291
Índice: 2, real=0, pred=0, prob=0.199
Índice: 3, real=0, pred=0, prob=0.416
Índice: 4, real=0, pred=0, prob=0.116


In [19]:
artifact = {
    'model_name': chosen_name,
    'pipeline': chosen_model,
    'threshold': selected_threshold,
    'feature_columns': X.columns.tolist(),
    'target': 'Churn',
    'positive_class': 1
}
joblib.dump(artifact, ARTIFACT_PATH)

metadata = {
    'model_name': chosen_name,
    'threshold': selected_threshold,
    'val_metrics': final_df.to_dict(),
    'test_metrics': test_metrics,
    'top_features': top_features
}
with METADATA_PATH.open('w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f'Archivo serializado: {ARTIFACT_PATH}')
print(f'Metadatos: {METADATA_PATH}')

Archivo serializado: c:\Tareas\Algoritmos\Proyecto\telco_model_final.joblib
Metadatos: c:\Tareas\Algoritmos\Proyecto\telco_model_metadata.json


## 9. Conclusiones

El proyecto desarrolló un flujo analítico y operativo reproducible para predecir el abandono de clientes a partir del conjunto de datos de Telco. A lo largo de la notebook, se consolidó una secuencia completa de trabajo: limpieza y validación de la fuente, ingeniería de variables orientada a la retención, preprocesamiento diferenciado por tipo de dato, comparación de modelos, validación cruzada, ajuste de hiperparámetros, optimización del umbral y serialización del pipeline final. Este enfoque permite que el modelo no sea solo un ejercicio de laboratorio, sino una herramienta de apoyo para la toma de decisiones en un contexto de negocio real.

La comparación entre referencia y modelos supervisados mostró que la métrica de accuracy por sí sola no es suficiente para evaluar un problema de retención. En este tipo de clasificación, un clasificador que predice mayoritariamente la clase dominante puede aparecer como competitivo desde una perspectiva superficial, pero resulta poco útil para detectar clientes en riesgo de abandono. Por ello, la selección del modelo se apoyó en criterios más informativos —especialmente F1-score, recall y ROC-AUC— en lugar de centrarse únicamente en la tasa global de aciertos.

El umbral de clasificación seleccionado se evaluó considerando el equilibrio entre la detección temprana de clientes en riesgo y la limitación de contactos innecesarios. En un entorno operativo real, este parámetro debe ajustarse con información de negocio: coste de no contactar a un cliente que abandona, coste de contactar a uno que no lo haría, y capacidad del equipo de retención. Por esta razón, el valor de 0.40 debe interpretarse como un punto de referencia técnico, no como una decisión final sin supervisión.

Desde una perspectiva metodológica, el modelo final debe entenderse como una herramienta de apoyo a la decisión y no como un mecanismo automático para imponer acciones comerciales o administrativas. La predicción ofrece una señal de riesgo, pero requiere validación humana, revisión de contexto y seguimiento continuo. Asimismo, las variables explicativas no deben interpretarse como factores causales absolutos: reflejan asociaciones aprendidas a partir del historial observado y deben contrastarse con el conocimiento del negocio.

En conjunto, la notebook presenta una solución técnica sólida, reproducible y documentada para la priorización de acciones de retención. Sus resultados son relevantes como evidencia analítica, siempre que se interpreten junto con las limitaciones del dataset, la naturaleza cambiante del comportamiento del cliente y la revisión responsable de las decisiones que se deriven del modelo.

Como trabajo futuro, se recomienda incorporar procesos de monitoreo de datos, evaluación de estabilidad temporal, auditoría por segmentos y revisión periódica del umbral. Estas medidas reforzarán la robustez del sistema y permitirán sostener su utilidad en entornos productivos con cambios reales en el comportamiento del cliente.


## 10. Entregables documentales

La notebook recoge el desarrollo reproducible del modelo, desde la preparación del dataset hasta la evaluación final y la simulación de perfiles de riesgo. A partir de ese análisis, se generan los documentos de presentación y acompañamiento que permiten comunicar los resultados a diferentes públicos.

Los entregables documentales del proyecto son los siguientes:

- [Ficha del modelo](ficha_modelo_telco.md): documento técnico y operativo que resume la finalidad del modelo, sus limitaciones, su alcance de uso y las condiciones para su aplicación.
- [Reporte ejecutivo](reporte_ejecutivo_telco.md): síntesis orientada a una audiencia no técnica, centrada en el problema, la decisión recomendada y el valor de negocio de la solución.
- [Informe técnico final](informe_tecnico_final_telco.md): documento de mayor profundidad que describa la metodología, resultados, riesgos, control de calidad y trazabilidad del proyecto.

Estos materiales deben actualizarse cada vez que cambien el dataset, el modelo, el umbral o las condiciones operativas de uso para mantener la consistencia documental y técnica de la entrega.


## 11. Auditoría complementaria de cumplimiento

Esta sección complementa la evaluación técnica del modelo con un conjunto de comprobaciones orientadas a la validación de cumplimiento, la trazabilidad y la robustez analítica del proyecto. Se incorporan elementos de comparación de estabilidad, análisis de rendimiento por kernel, revisión de curvas de evaluación, matriz de confusión, diagnóstico de errores locales, auditoría por grupos y simulación de perfiles de riesgo con registros nuevos.

El propósito de esta auditoría no es únicamente verificar la calidad del modelo desde una perspectiva estadística, sino también confirmar que la solución mantiene una estructura coherente con los requisitos de entrega, la interpretación del negocio y la posible presentación final ante una audiencia técnica o académica.


In [20]:
# 8) Cinco perfiles de inferencia sobre registros raw.
from inferencia_telco import predict_record

base_profile = {
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'No',
    'Dependents': 'No',
    'tenure': 24,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'Yes',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 80.0,
    'TotalCharges': '1920',
    'Churn': 'Yes',
}

profiles = {
    'bajo_riesgo': {**base_profile, 'tenure': 60, 'Contract': 'Two year', 'MonthlyCharges': 45.0, 'TotalCharges': '2700'},
    'riesgo_medio': {**base_profile, 'tenure': 24, 'Contract': 'One year', 'MonthlyCharges': 65.0, 'TotalCharges': '1560'},
    'alto_riesgo': {**base_profile},
    'categoria_poco_frecuente': {**base_profile, 'InternetService': 'DSL', 'PaymentMethod': 'Credit card (automatic)'},
    'cercano_umbral': {**base_profile, 'tenure': 12, 'MonthlyCharges': 55.0, 'TotalCharges': '660'},
}

profile_rows = []
for profile_name, profile in profiles.items():
    result = predict_record(profile)
    profile_rows.append({
        'Perfil': profile_name,
        'Probabilidad': result['probabilidad_abandono'],
        'Clasificación': result['clasificacion'],
        'Umbral': result['umbral'],
        'Versión': result['version'],
        'Revisión humana': result['revision_humana'],
    })

profiles_df = pd.DataFrame(profile_rows)
display(profiles_df)


,Perfil,Probabilidad,Clasificación,Umbral,Versión,Revisión humana
0,bajo_riesgo,0.233432,Riesgo bajo,0.4,1.0,True
1,riesgo_medio,0.279936,Riesgo bajo,0.4,1.0,True
2,alto_riesgo,0.653671,Riesgo de abandono,0.4,1.0,True
3,categoria_poco_frecuente,0.418318,Riesgo de abandono,0.4,1.0,True
4,cercano_umbral,0.552006,Riesgo de abandono,0.4,1.0,True
